# Matatu Association: Demand & Fleet Planning Analysis

Reusable logic lives in `src/`; this notebook only orchestrates and displays results.

| Module | Contents |
|---|---|
| `src/routes.py` | `Route` class, route data |
| `src/equilibrium.py` | Supply-demand equilibrium (linear solve) |
| `src/forecasters.py` | `Forecaster` base + Moving Average, SES, Linear Trend, Seasonal-Naive |
| `src/backtest.py` | Rolling-origin MAE, alpha grid search, model selection |
| `src/fleet.py` | Fleet sizing |
| `src/seasonality.py` | 60-day weekly-seasonality simulation and backtest |
| `src/plotting.py` | Plot helpers |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.routes import load_routes
from src.equilibrium import solve_equilibrium, interpret_fare
from src.backtest import evaluate_route
from src.fleet import vehicles_needed, TRIPS_PER_DAY, SEATS_PER_VEHICLE, BUFFER
from src.seasonality import simulate_counts, backtest_seasonal, WEEKDAY_EFFECT
from src.plotting import plot_forecast_vs_actual, plot_seasonality

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
%matplotlib inline

## 1. Route statistics & revenue

In [ ]:
routes = load_routes()

rows = []
for r in routes.values():
    s = r.descriptive_stats()
    rows.append({"Route": r.name, "Fare (UGX)": r.fare, "Mean pax/day": s["mean"],
                 "Variance": s["variance"], "Std dev": s["stdev"],
                 "Min": s["min"], "Max": s["max"], "Total revenue (UGX)": r.total_revenue()})
pd.DataFrame(rows).set_index("Route").round(2)

## 2. Supply-demand equilibrium (Kampala-Ntinda)

$Q_d = 120 - 0.02P$, $Q_s = 10 + 0.03P$, solved as a linear system.

In [ ]:
q_eq, p_eq = solve_equilibrium()
print(f"Equilibrium fare P*     = UGX {p_eq:,.2f}")
print(f"Equilibrium quantity Q* = {q_eq:,.2f} passengers/trip-hour")
print(interpret_fare(routes["Kampala-Ntinda"].fare, p_eq))

## 3-4. Rolling-origin backtest (days 4-10) and alpha grid search

In [ ]:
TEST_DAYS = list(range(4, 11))

results = {name: evaluate_route(r.passenger_counts, TEST_DAYS) for name, r in routes.items()}

for name, res in results.items():
    print(f"{name}: best alpha = {res['best_alpha']}")

mae_table = pd.DataFrame({n: res["maes"] for n, res in results.items()}).T
mae_table["Best model"] = [res["best_model"] for res in results.values()]
mae_table

In [ ]:
alpha_table = pd.DataFrame({n: res["alpha_maes"] for n, res in results.items()})
alpha_table.index.name = "alpha"
alpha_table.round(2)

## 5. Day-11 revenue forecast

In [ ]:
forecasts = {n: res["forecast"] for n, res in results.items()}
best_models = {n: res["best_model"] for n, res in results.items()}

pd.DataFrame({
    "Best model": best_models,
    "Day-11 passengers": {n: forecasts[n] for n in routes},
    "Day-11 revenue (UGX)": {n: forecasts[n] * routes[n].fare for n in routes},
}).round(1)

In [ ]:
fig = plot_forecast_vs_actual(routes, forecasts, best_models,
                              save_path=OUT / "actual_vs_forecast_revenue.png")
plt.show()

## 6. Fleet planning for day 11

Assumptions: 8 trips/day, 14 seats/vehicle, 15% buffer.

In [ ]:
fleet = pd.DataFrame({n: vehicles_needed(forecasts[n]) for n in routes}).T
fleet.columns = ["Forecast pax", "Pax with 15% buffer", "Capacity/vehicle", "Vehicles needed"]
fleet = fleet.round(1)
fleet["Vehicles needed"] = fleet["Vehicles needed"].astype(int)
fleet

## Extension: weekly seasonality (60 days)

Busy Fridays, quiet Sundays. Seasonal-Naive (same weekday last week) vs 3-day Moving Average.

In [ ]:
counts = simulate_counts(n_days=60, base=50, noise_sd=3.0, seed=42)
ma_mae, sn_mae, _, _ = backtest_seasonal(counts, period=7, ma_window=3)

print(f"Weekday effects: {WEEKDAY_EFFECT}")
print(f"Moving Average(3)  MAE = {ma_mae:.2f}")
print(f"Seasonal-Naive(7)  MAE = {sn_mae:.2f}")
print(f"Seasonal-Naive improves MAE by {(1 - sn_mae / ma_mae) * 100:.1f}%")

In [ ]:
fig = plot_seasonality(counts, ma_mae, sn_mae, save_path=OUT / "extension_weekly_seasonality.png")
plt.show()